In [ ]:
# 01_build_chunks.ipynb
# Цель:
# 1) прочитать DATA.zip
# 2) извлечь текст по страницам
# 3) нарезать каждую страницу на чанки
# 4) присвоить стабильные chunk_id
# 5) сохранить chunks_final.csv
#
# ВАЖНО:
# - chunk_id должны быть детерминированными
# - порядок обхода компаний и страниц должен быть фиксированным
# - после согласования chunks_final.csv не меняем

In [ ]:
import re
import zipfile
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
DATA_ZIP_PATH = "DATA.zip"
OUTPUT_PATH = "chunks_final.csv"

CHUNK_SIZE = 300
CHUNK_OVERLAP = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    is_separator_regex=False,
)

In [ ]:
def make_company_slug(name: str) -> str:
    """
    Преобразует исходное название файла/компании в стабильный slug.
    
    Примеры:
    'ПАО МТС.pdf' -> 'мтс'
    'АО «Лента».pdf' -> 'лента'
    """
    # TODO: реализовать
    raise NotImplementedError

# Ожидаемая логика:
# 1. убрать расширение файла
# 2. привести к нижнему регистру
# 3. убрать юр. формы: пао, ао, мкпао, ao, pao
# 4. убрать кавычки/спецсимволы
# 5. схлопнуть повторные пробелы
# 6. strip()

In [ ]:
# Тест
test_names = [
    "ПАО МТС.pdf",
    "АО «Лента».pdf",
    "МКПАО Хэдхантер.pdf",
]

for name in test_names:
    print(name, "->", make_company_slug(name))

In [ ]:
# Тащим сюда логику из read_pdf.py и chanking.py, адаптируем.

def extract_text_by_page_from_pdf_bytes(pdf_bytes: bytes) -> dict[int, str]:
    """
    Возвращает словарь:
    {номер_страницы_с_1: текст_страницы}
    """
    # TODO: реализовать
    raise NotImplementedError

def load_reports_from_zip(zip_path: str) -> dict[str, dict[int, str]]:
    """
    Читает DATA.zip и возвращает структуру:
    {
        source_file_name: {
            1: 'текст страницы 1',
            2: 'текст страницы 2',
            ...
        },
        ...
    }
    """
    # TODO: реализовать
    raise NotImplementedError

In [ ]:
# Проверка, например так
reports = load_reports_from_zip(DATA_ZIP_PATH)

print(f"Число файлов: {len(reports)}")
for i, (file_name, pages) in enumerate(sorted(reports.items())):
    print(file_name, "->", len(pages), "pages")
    if i == 4:
        break

In [ ]:
# Функция для получения чанков из текста страницы
def split_page_text(page_text: str, splitter) -> list[str]:
    """
    Нарезает текст страницы на чанки.
    Пустые/мусорные чанки нужно отфильтровать.
    """
    # TODO: реализовать
    raise NotImplementedError

# не возвращать пустые строки;
# обрезать лишние пробелы;
# сохранять порядок чанков.

In [ ]:
# Проверка 
sample_file = sorted(reports.keys())[0]
sample_page = sorted(reports[sample_file].keys())[0]
sample_text = reports[sample_file][sample_page]

sample_chunks = split_page_text(sample_text, splitter)
print("Chunks on sample page:", len(sample_chunks))
for i, ch in enumerate(sample_chunks[:3], start=1):
    print("=" * 40)
    print(i, len(ch))
    print(ch[:400])

In [ ]:
# Основная функция ноутбука
def build_chunks_dataframe(
    reports: dict[str, dict[int, str]],
    splitter
) -> pd.DataFrame:
    """
    Собирает единый DataFrame чанков со столбцами:
    - chunk_id
    - company_raw
    - company_slug
    - source_file
    - pdf_page
    - chunk_idx_on_page
    - text
    - n_chars
    """
    # TODO: реализовать
    raise NotImplementedError

# Логика:
# 1. идти по файлам в sorted(...)
# 2. для каждого файла получить company_slug = make_company_slug(...)
# 3. идти по страницам в sorted(...)
# 4. разбить страницу на chunks
# 5. chunk_idx_on_page начинать с 1 на каждой странице
# 6. chunk_id = f"{company_slug}-p{page_num:03d}-c{chunk_idx:03d}"
# 7. собрать rows
# 8. вернуть pd.DataFrame(rows)

In [ ]:
chunks_df = build_chunks_dataframe(reports, splitter)
chunks_df.head()

In [ ]:
# Пример проверки качества
def validate_chunks_dataframe(df: pd.DataFrame) -> None:
    required_cols = [
        "chunk_id",
        "company_raw",
        "company_slug",
        "source_file",
        "pdf_page",
        "chunk_idx_on_page",
        "text",
        "n_chars",
    ]
    
    missing = [c for c in required_cols if c not in df.columns]
    assert not missing, f"Не хватает колонок: {missing}"
    
    assert df["chunk_id"].is_unique, "chunk_id должны быть уникальны"
    assert (df["pdf_page"] >= 1).all(), "Номер страницы должен быть >= 1"
    assert (df["chunk_idx_on_page"] >= 1).all(), "chunk_idx_on_page должен быть >= 1"
    assert df["text"].fillna("").str.strip().ne("").all(), "Есть пустые чанки"
    assert (df["n_chars"] == df["text"].str.len()).all(), "n_chars не совпадает с длиной text"
    
    duplicated_inside_page = df.duplicated(
        subset=["company_slug", "pdf_page", "chunk_idx_on_page"]
    )
    assert not duplicated_inside_page.any(), \
        "Есть дубли chunk_idx_on_page внутри company_slug + pdf_page"
    
    print("Validation passed.")


validate_chunks_dataframe(chunks_df)

In [ ]:
# Ещё посмотрите на результаты
print("Всего чанков:", len(chunks_df))
print("Компаний:", chunks_df["company_slug"].nunique())

chunks_df[[
    "chunk_id", "company_slug", "pdf_page", "chunk_idx_on_page", "n_chars"
]].head(20)

In [ ]:
chunks_df.groupby("company_slug").size().sort_values(ascending=False).head(10)

In [ ]:
chunks_df.groupby(["company_slug", "pdf_page"]).size().head(20)

In [ ]:
# Сохраняем результат
chunks_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Saved to {OUTPUT_PATH}")

In [ ]:
# Проверяем сохранённый файл
check_df = pd.read_csv(OUTPUT_PATH)
print(check_df.shape)
check_df.head()

# После этого полученный csv файл коммитим в репозиторий
# и не меняем, чтобы на него можно было ссылаться в следующих шагах.